# Content Moderation System - Complete Training Notebook
## Phases 1, 2, and 3 - Optimized for Google Colab GPU

This notebook trains all three phases:
- **Phase 1**: Text baseline classifier (6 toxic categories)
- **Phase 2**: Multilingual support (XLM-RoBERTa)
- **Phase 3**: Vision & OCR (NSFW, hate symbols, violence detection)

**Total training time on Colab GPU**: ~2-3 hours

## 📋 Table of Contents
1. Setup & GPU Check
2. Clone Repository
3. Install Dependencies
4. Phase 1: Text Baseline Training
5. Phase 2: Multilingual Training
6. Phase 3: Vision & OCR Training
7. Final Evaluation & Export

## 1️⃣ Setup & Check GPU

In [ ]:
# Check if GPU is available
import torch
print("🔍 GPU Check:")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: GPU not detected. Training will be slow!")
    print("   Go to Runtime → Change runtime type → GPU")

## 2️⃣ Clone Repository

In [ ]:
# Mount Google Drive (optional - for saving models)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

In [ ]:
# Clone repository
import os
os.chdir('/content')

!git clone https://github.com/hschinmayabharadwaj/Content-Moderation.git
os.chdir('/content/Content-Moderation/content-moderation-system')
print("✅ Repository cloned!")
print(f"Current directory: {os.getcwd()}")

## 3️⃣ Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q torch torchvision transformers datasets pyyaml pandas numpy scikit-learn tqdm -U
print("✅ Core dependencies installed")

In [ ]:
# Install Phase 3 specific dependencies
!pip install -q easyocr opencv-python scikit-image albumentations timm gradio imagehash -U
print("✅ Phase 3 dependencies installed")
print("📥 EasyOCR models downloading (this may take a moment)...")

## 4️⃣ Phase 1: Text Baseline Training

In [ ]:
print("🚀 PHASE 1: TEXT BASELINE TRAINING")
print("="*80)
print("Training: Multi-label toxic comment classifier")
print("Model: DistilBERT")
print("Categories: toxic, severe_toxic, obscene, threat, insult, identity_hate")
print("\nDownloading dataset...")

os.chdir('/content/Content-Moderation/content-moderation-system/phase1_text_baseline')
!python download_data.py --output-dir data --create-sample --sample-size 5000
print("✅ Dataset prepared")

In [ ]:
# Train Phase 1 model
print("\n🎯 Training Phase 1 model...\n")
!python train_classifier.py --config configs/baseline.yaml --use-sample --verbose
print("\n✅ Phase 1 training complete!")

In [ ]:
# Calibrate Phase 1 thresholds
print("📊 Calibrating thresholds...\n")
!python calibrate_thresholds.py --config configs/baseline.yaml
print("✅ Phase 1 calibration complete!")

## 5️⃣ Phase 2: Multilingual Training

In [ ]:
print("\n🚀 PHASE 2: MULTILINGUAL TRAINING")
print("="*80)
print("Training: Multilingual toxic comment classifier")
print("Model: XLM-RoBERTa")
print("Languages: English + Multilingual support")

os.chdir('/content/Content-Moderation/content-moderation-system/phase2_multilingual')
print("\nPreparing multilingual datasets...")
!python prepare_datasets.py --config configs/xlm_roberta.yaml
print("✅ Multilingual datasets prepared")

In [ ]:
# Train Phase 2 model
print("\n🎯 Training Phase 2 multilingual model...\n")
!python train_multilingual.py --config configs/xlm_roberta.yaml --verbose
print("\n✅ Phase 2 training complete!")

In [ ]:
# Evaluate Phase 2
print("📊 Evaluating Phase 2 model...\n")
!python evaluate_multilingual.py --config configs/xlm_roberta.yaml --model-path models/best_model.pt
print("✅ Phase 2 evaluation complete!")

## 6️⃣ Phase 3: Vision & OCR Training

In [ ]:
print("\n🚀 PHASE 3: VISION & OCR TRAINING")
print("="*80)
print("Training: Image classifiers for content moderation")
print("Models:")
print("  1. NSFW Detection (safe vs explicit)")
print("  2. Hate Symbols Detection (no_hate vs hate)")
print("  3. Violence Detection (no_violence vs violence)")

os.chdir('/content/Content-Moderation/content-moderation-system/phase3_vision_ocr')
print("\nCreating dummy datasets (1,440 images)...")
!python scripts/download_datasets.py --dummy --num-samples 200
print("✅ Datasets created")

In [ ]:
# Train NSFW classifier
print("\n🎯 Training NSFW detector...\n")
!python scripts/train_image_model.py --dataset nsfw --backbone resnet50 --epochs 15 --batch-size 32 --verbose
print("✅ NSFW model trained!")

In [ ]:
# Train hate symbols classifier
print("\n🎯 Training hate symbols detector...\n")
!python scripts/train_image_model.py --dataset hate_symbols --backbone resnet50 --epochs 15 --batch-size 32 --verbose
print("✅ Hate symbols model trained!")

In [ ]:
# Train violence classifier
print("\n🎯 Training violence detector...\n")
!python scripts/train_image_model.py --dataset violence --backbone efficientnet_b0 --epochs 15 --batch-size 32 --verbose
print("✅ Violence model trained!")

## 7️⃣ Final Evaluation & Export

In [ ]:
# Test OCR
print("\n🔍 Testing OCR System...\n")
!python scripts/create_test_images.py
!python scripts/test_ocr.py --batch test_images/ 2>/dev/null | tail -20
print("✅ OCR testing complete!")

In [ ]:
# Full multimodal evaluation
print("\n🧪 Running full multimodal system test...\n")
!python scripts/test_multimodal.py 2>/dev/null | tail -40
print("✅ Multimodal testing complete!")

## 📦 Export Models & Results

In [ ]:
# Create summary of all trained models
import json
from pathlib import Path

print("\n📊 TRAINING SUMMARY")
print("="*80)

results = {
    "phases": {},
    "timestamp": str(pd.Timestamp.now()),
    "gpu_info": {
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    }
}

# Phase 1 models
phase1_dir = Path('/content/Content-Moderation/content-moderation-system/phase1_text_baseline/models')
if phase1_dir.exists():
    results["phases"]["phase1"] = {
        "status": "✅ Trained",
        "model_path": str(phase1_dir / 'best_model.pt'),
        "files": [f.name for f in phase1_dir.glob('*')]
    }

# Phase 2 models
phase2_dir = Path('/content/Content-Moderation/content-moderation-system/phase2_multilingual/models')
if phase2_dir.exists():
    results["phases"]["phase2"] = {
        "status": "✅ Trained",
        "model_path": str(phase2_dir / 'best_model.pt'),
        "files": [f.name for f in phase2_dir.glob('*')]
    }

# Phase 3 models
phase3_dir = Path('/content/Content-Moderation/content-moderation-system/phase3_vision_ocr/models')
if phase3_dir.exists():
    results["phases"]["phase3"] = {
        "status": "✅ Trained",
        "models": {}
    }
    for model_type in ['nsfw', 'hate_symbols', 'violence']:
        model_dir = phase3_dir / model_type
        if model_dir.exists():
            results["phases"]["phase3"]["models"][model_type] = {
                "model_path": str(model_dir / 'best_model.pt'),
                "files": [f.name for f in model_dir.glob('*')]
            }

print(json.dumps(results, indent=2))

# Save summary
summary_path = '/content/Content-Moderation/content-moderation-system/TRAINING_SUMMARY.json'
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Summary saved to: {summary_path}")

In [ ]:
# Copy models to Google Drive for backup
import shutil

print("\n💾 Backing up models to Google Drive...\n")

backup_dir = '/content/drive/My Drive/ContentModeration_TrainedModels'
os.makedirs(backup_dir, exist_ok=True)

# Backup Phase 1
phase1_models = '/content/Content-Moderation/content-moderation-system/phase1_text_baseline/models'
if os.path.exists(phase1_models):
    shutil.copytree(phase1_models, f'{backup_dir}/phase1_models', dirs_exist_ok=True)
    print("✅ Phase 1 models backed up")

# Backup Phase 2
phase2_models = '/content/Content-Moderation/content-moderation-system/phase2_multilingual/models'
if os.path.exists(phase2_models):
    shutil.copytree(phase2_models, f'{backup_dir}/phase2_models', dirs_exist_ok=True)
    print("✅ Phase 2 models backed up")

# Backup Phase 3
phase3_models = '/content/Content-Moderation/content-moderation-system/phase3_vision_ocr/models'
if os.path.exists(phase3_models):
    shutil.copytree(phase3_models, f'{backup_dir}/phase3_models', dirs_exist_ok=True)
    print("✅ Phase 3 models backed up")

# Backup training summary
shutil.copy(summary_path, f'{backup_dir}/TRAINING_SUMMARY.json')
print("✅ Training summary backed up")

print(f"\n📁 All backups saved to: {backup_dir}")

## ✅ Training Complete!

In [ ]:
print("\n" + "="*80)
print("🎉 ALL PHASES TRAINED SUCCESSFULLY!")
print("="*80)

print("\n📊 Summary:")
print("  ✅ Phase 1: Text Baseline Classifier")
print("     - Model: DistilBERT")
print("     - Categories: 6 (toxic, severe_toxic, obscene, threat, insult, identity_hate)")
print("\n  ✅ Phase 2: Multilingual Classifier")
print("     - Model: XLM-RoBERTa")
print("     - Support: English + Multilingual")
print("\n  ✅ Phase 3: Vision & OCR")
print("     - NSFW Detector (ResNet50)")
print("     - Hate Symbols Detector (ResNet50)")
print("     - Violence Detector (EfficientNet)")
print("     - OCR: EasyOCR")

print("\n📁 Model Locations:")
print("  - Phase 1: phase1_text_baseline/models/best_model.pt")
print("  - Phase 2: phase2_multilingual/models/best_model.pt")
print("  - Phase 3: phase3_vision_ocr/models/{nsfw,hate_symbols,violence}/best_model.pt")

print("\n💾 Backups:")
print(f"  - Saved to Google Drive: {backup_dir}")

print("\n🚀 Next Steps:")
print("  1. Download models from Google Drive")
print("  2. Use multimodal_moderator.py for inference")
print("  3. Deploy to production")

print("\n" + "="*80)

## 📚 Optional: Test Inference

In [ ]:
# Optional: Test inference on a sample
import sys
sys.path.insert(0, '/content/Content-Moderation/content-moderation-system')

print("\n🧪 Testing Phase 1 Inference:\n")
# Example text moderation
from text_moderator import TextModerator

try:
    moderator = TextModerator(models_dir='content_moderation_trained')
    result = moderator.check_text("This is a wonderful day!")
    moderator.display_results(result)
except Exception as e:
    print(f"Note: {e}")
    print("(This is expected if models_dir path needs adjustment)")